Data Governance Architecture Assistant (LLM-OpenAI-GPT4o-RAG)

This solution has been developed for a well-known software development company who is looking at implementing LLMs into their enterprise cloud Data Governance platform to provide guidance to data enablers and practitioners (data stewards and data architects) who define and operationalize the governance framework. 

It was built implementing Retrieval Augmented Generation (RAG) to create the context-aware chatbot on current company's policy and procedures related to Data Governance Architecture.

This file is the SandBox version with output on this same Jupyter Notebook environment.

# Query Samples
Which tables shall be used in
which cases, and what are the benefits of using managed tables?

Who created Data Governance Architecture Assistant?

In [6]:
# Loading libraries
import os
import shutil
from dotenv import load_dotenv
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Loading OpenAI API Key
load_dotenv() 

# Enabling Caching
# This line should be change for enterpise cloud developments (Distributed Caching is the Enterprise Standard)
set_llm_cache(SQLiteCache(database_path=".langchain.db"))

# Initialization
def initialize_rag_system():
    # Loading Data
    file_path = "data-governance-architecture-patterns.pdf"
    persist_directory = "./chroma_db_gov"

    # Cleanup warning
    # If the folder exists, deletion ensure use OpenAI's 1536 dimensions
    if os.path.exists(persist_directory):
        try:
            shutil.rmtree(persist_directory)
            print("Successfully cleared old database.")
        except Exception as e:
            print(f"Warning: Could not auto-delete database. Manual delete required: {e}")
 

    loader = PyPDFLoader(file_path)
    governance_docs = loader.load()

    # OpenAI Models
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    # Split & Index (Using standard parameters for technical RAG)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(governance_docs)
    
    # Re-initialize with standard OpenAI dimensions
    # ChromaDB vector database runs locally (Open source and free) unlike those vector databases that require 
    # a cloud account and API keys setting up a server (e.g., Pinecone).
    vectorstore = Chroma.from_documents(
        documents=splits, 
        embedding=embeddings,
        persist_directory=persist_directory
    )
    # Probabilistic sampling Top-K
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
     # Grounding Prompt: Data Governance Architecture context
    template = """You are an expert Data Governance Architecture Assistant. 
    Use the following retrieved context from the architectural documentation to answer the question.
    
    ### CONSTRAINTS:
    - Do NOT start your response with phrases like "According to the context" or "Based on the documents."
    - Start the answer directly.
    - If the answer is NOT in the context, respond: "There is no information in our current policy and procedures to answer your question. Please contact us directly at datagovernance@mycompany.com."
    - Keep answers technical and precise.

    Context: {context}
    Question: {question}
    Answer:"""
    prompt = ChatPromptTemplate.from_template(template)

    # Building Chain
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

if __name__ == "__main__":
    try:
        # Initializing Chain
        rag_chain = initialize_rag_system()
        # Executiting Sample
        query = "Who created Data Governance Architecture Assistant?"
        print("\nRESULT:\n", rag_chain.invoke(query))
    except Exception as e:
        print(f"Final Error: {e}")


RESULT:
 There is no information in our current policy and procedures to answer your question. Please contact us directly at datagovernance@mycompany.com.


Note: Warning given before RESULT section is caused by the ChromaDB engine (via LangChain) remains active in local computer's memory, preventing Windows from deleting the database files even after the script finishes.
To resolve this without restarting local computer every time, must manually clear the old directory one last time and then use a randomized directory approach to avoid future locks (use a unique folder name to avoid "File in Use" errors).
In a production environment, best practice keeps one static name, but for development/debugging (local computer), this prevents locking issues.